# thui-lora-train — LoRA SFT on winning duck-harness turns

Trains a rank-16 LoRA (q/v of the 16 full-attention layers) on 1,826 single-turn
samples extracted from levels that duck/thui runs actually CLEARED. Held-out games
are excluded from training and listed in the output for the eval kernel.

This kernel scores nothing and submits nothing. Output: `/kaggle/working/adapter/`.

Solver credit: the data derives from runs of the Tufa Labs duck harness
(Bessis, Cottaar, Pressman, Smit, Tesnar, Viel). Knowless Crew / Thuitanium fork.


In [ ]:
import glob, os, subprocess, sys

def find(*pats):
    for p in pats:
        hits = glob.glob(p)
        if hits:
            return hits[0]
    return None

DATA = find("/kaggle/input/datasets/sahasawatt/thui-lora-train-v1",
            "/kaggle/input/thui-lora-train-v1")
assert DATA, "training dataset not mounted"
WHEELS = find("/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3",
              "/kaggle/input/arc3-vllm-h100-wheelhouse-v3")
assert WHEELS, "wheelhouse not mounted"
print("DATA =", DATA)
print("WHEELS =", WHEELS)


In [ ]:
import glob, subprocess, sys, zipfile, os
libs = os.path.join(DATA, "libs")
if not os.path.isdir(libs):
    z = os.path.join(DATA, "libs.zip")
    assert os.path.exists(z), "libs dir/zip missing from dataset"
    os.makedirs("/kaggle/working/libs", exist_ok=True)
    with zipfile.ZipFile(z) as zh:
        zh.extractall("/kaggle/working/libs")
    libs = "/kaggle/working/libs"
wheels = glob.glob(os.path.join(libs, "*.whl"))
assert wheels, "no wheels found"
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
# base image ships torchao 0.10; transformers 5.16 hard-fails on import if it sees <0.16.
# It is an OPTIONAL integration -- absent is fine, old is fatal.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index",
                       "--no-deps", *wheels])
import peft
print("peft", peft.__version__)


In [ ]:
import os, shutil, subprocess, sys
src = os.path.join(DATA, "train_lora.py")
if not os.path.exists(src):
    cand = "/kaggle/working/train_lora.py"
else:
    cand = src
data_dir = os.path.join(DATA, "data")
if not os.path.isdir(data_dir):
    import zipfile
    dz = os.path.join(DATA, "data.zip")
    assert os.path.exists(dz), "data dir/zip missing"
    os.makedirs("/kaggle/working/data_unzip", exist_ok=True)
    with zipfile.ZipFile(dz) as zh:
        zh.extractall("/kaggle/working/data_unzip")
    staged = "/kaggle/working/staged"
    os.makedirs(os.path.join(staged, "data"), exist_ok=True)
    shutil.copy(os.path.join("/kaggle/working/data_unzip", "sft-all.jsonl"),
                os.path.join(staged, "data", "sft-all.jsonl"))
    shutil.copy(cand, os.path.join(staged, "train_lora.py"))
    os.environ["THUI_DATA_OVERRIDE"] = staged
    print("staged unzipped data at", staged)
print("train script:", cand)


In [ ]:
import os, subprocess, sys
env = dict(os.environ)
# offline HF kernel cache: the fp8 forward needs kernels-community/finegrained-fp8
# (244KB of Triton source). Shipped in the dataset; staged writable because the hub
# library takes locks even for cache reads.
import zipfile as _zf
hz = os.path.join(DATA, "hfcache.zip")
hdir = os.path.join(DATA, "hfcache")
if os.path.exists(hz):
    with _zf.ZipFile(hz) as z:
        z.extractall("/kaggle/working/hfcache")
    hdir = "/kaggle/working/hfcache"
assert os.path.isdir(hdir), "hfcache missing from dataset"
env["HF_HOME"] = hdir
env["HF_HUB_OFFLINE"] = "1"
env.setdefault("MAX_STEPS", "0")    # FULL RUN. Smoke (v12) proved ladder+loop+save at 10 steps.
env.setdefault("EPOCHS", "2")       # 75s/step at MAX_LEN 4096 (v14) -> 2 epochs x ~166 steps ~ 7h, fits the cap
env.setdefault("RANK", "16")
env.setdefault("MAX_LEN", "4096")   # v13 OOM at step 60: 7.58GiB alloc, 2.15 free -- logits+activations at 8192 bust 95GB
env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # v13 also fragmented across 60 varying-length steps
script = os.path.join(DATA, "train_lora.py")
if not os.path.exists(script):
    script = "/kaggle/working/staged/train_lora.py"
r = subprocess.run([sys.executable, script], env=env)
assert r.returncode == 0, f"training exited {r.returncode}"
print("TRAIN-OK")
